In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

# PySpark SQL Functions:
- `createOrReplaceTempView`
- `createOrReplaceGlobalTempView`

These functions allow you to register a DataFrame as a temporary SQL table,
enabling you to query it using SQL syntax.

### Create DataFrame

In [73]:
# Sample data
employees_data = [
    (1, 'John Doe', 'Engineering', 75000, '2020-01-15'),
    (2, 'Jane Smith', 'Marketing', 65000, '2019-03-20'),
    (3, 'Bob Johnson', 'Engineering', 80000, '2021-06-10'),
    (4, 'Alice Williams', 'HR', 60000, '2018-11-05'),
    (5, 'Charlie Brown', 'Engineering', 70000, '2022-02-28')
]

In [64]:
df = spark.createDataFrame(data = employees_data, schema = (['emp_id', 'name', 'department', 'salary', 'join_date']))
df.show()

+------+--------------+-----------+------+----------+
|emp_id|          name| department|salary| join_date|
+------+--------------+-----------+------+----------+
|     1|      John Doe|Engineering| 75000|2020-01-15|
|     2|    Jane Smith|  Marketing| 65000|2019-03-20|
|     3|   Bob Johnson|Engineering| 80000|2021-06-10|
|     4|Alice Williams|         HR| 60000|2018-11-05|
|     5| Charlie Brown|Engineering| 70000|2022-02-28|
+------+--------------+-----------+------+----------+



## Create local temp view

### `createOrReplaceTempView`
- Creates a temporary view that is session-scoped
- The view will be dropped when the session ends

In [110]:
df.createOrReplaceTempView('employee')

In [74]:
# Use the created view to query the data
result = spark.sql('SELECT * FROM employee WHERE salary > 65000')
result.show()

+------+-------------+-----------+------+----------+
|emp_id|         name| department|salary| join_date|
+------+-------------+-----------+------+----------+
|     1|     John Doe|Engineering| 75000|2020-01-15|
|     3|  Bob Johnson|Engineering| 80000|2021-06-10|
|     5|Charlie Brown|Engineering| 70000|2022-02-28|
+------+-------------+-----------+------+----------+



In [83]:
result = spark.sql('SELECT department, COUNT(*) AS emp_count FROM employee GROUP BY ALL ORDER BY 1')
result.show()

+-----------+---------+
| department|emp_count|
+-----------+---------+
|Engineering|        3|
|         HR|        1|
|  Marketing|        1|
+-----------+---------+



## Create global temp view

### `createOrReplaceGlobalTempView`
- Creates a global temporary view that is application-scoped
- Shared across all sessions within the same application
- Must be accessed using `global_temp` database prefix
- The view will be dropped when the application ends

In [111]:
df.createOrReplaceGlobalTempView('global_employee')

In [90]:
result = spark.sql('SELECT * FROM global_temp.global_employee WHERE join_date < "2021-01-01"')
result.show()

+------+--------------+-----------+------+----------+
|emp_id|          name| department|salary| join_date|
+------+--------------+-----------+------+----------+
|     1|      John Doe|Engineering| 75000|2020-01-15|
|     2|    Jane Smith|  Marketing| 65000|2019-03-20|
|     4|Alice Williams|         HR| 60000|2018-11-05|
+------+--------------+-----------+------+----------+



In [87]:
# Clean up (optional)
spark.catalog.dropTempView("employee")
spark.catalog.dropGlobalTempView("global_employee")

True